# Применение pipeline

<div class="alert alert-info">

<b>Про Задачу </b>

Попробуем классифицировать людей по оценкам уровня счастья
    
</div>

## Imports

In [81]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

from sklearn.metrics import f1_score, make_scorer


## Загрузка данных

<div class="alert alert-info">

<b>Про датасет </b>


    
Приведеные результаты опроса молодежи до 21-го года по тому, какие вещи делют их счастливыми по 5-й шкале
    
</div>

In [82]:
file_path = 'https://raw.githubusercontent.com/a-milenkin/Datasetes_for_Piplines/main/responses.csv'
data = pd.read_csv(file_path)

In [83]:
data.head(3)

,Music,Slow songs or fast songs,Dance,Folk,Country,Classical music,Musical,Pop,Rock,Metal or Hardrock,...,Age,Height,Weight,Number of siblings,Gender,Left - right handed,Education,Only child,Village - town,House - block of flats
0,5.0,3.0,2.0,1.0,2.0,2.0,1.0,5.0,5.0,1.0,...,20.0,163.0,48.0,1.0,female,right handed,college/bachelor degree,no,village,block of flats
1,4.0,4.0,2.0,1.0,1.0,1.0,2.0,3.0,5.0,4.0,...,19.0,163.0,58.0,2.0,female,right handed,college/bachelor degree,no,city,block of flats
2,5.0,5.0,2.0,2.0,3.0,4.0,5.0,3.0,5.0,3.0,...,20.0,176.0,67.0,2.0,female,right handed,secondary school,no,city,block of flats


In [84]:
data.shape

(1010, 150)

Выделим только часть данных, для простоты работы

In [85]:
interesting_features = ["Age", "Height", "Gender", "Weight", 
              "Left - right handed", "Village - town", "Getting up", "God",
              "Health", "Borrowed stuff", "Self-criticism", "Elections", 
              "Smoking", "Alcohol" ,"Number of friends", "Spending on healthy eating",
              "Music", "Movies", "Friends versus money", "Changing the past"]


target = "Happiness in life"

In [86]:
data.dropna(inplace=True)
X = data[interesting_features]
y = data[target]

In [87]:
# Сбалансируем классы

def happiness_score(x):
    if x < 4:
        return -1
    elif x == 4.0:
        return 0
    else:
        return 1

In [88]:
y = y.apply(happiness_score)

In [89]:
X.dtypes

Age                           float64
Height                        float64
Gender                         object
Weight                        float64
Left - right handed            object
Village - town                 object
Getting up                    float64
God                           float64
Health                        float64
Borrowed stuff                float64
Self-criticism                float64
Elections                     float64
Smoking                        object
Alcohol                        object
Number of friends               int64
Spending on healthy eating    float64
Music                         float64
Movies                        float64
Friends versus money          float64
Changing the past             float64
dtype: object

## Разделим данные

<div class="alert alert-warning">

<b>Разделение переменных</b>

Автоматическое разделение может привести к тому, что если числа были сохранены как строки, то вы получите неверный датафрейм

<b>Рекомендую проверять данные "глазами"</b>
</div>

In [90]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

In [91]:
X[categorical_features].head()

,Gender,Left - right handed,Village - town,Smoking,Alcohol
0,female,right handed,village,never smoked,drink a lot
1,female,right handed,city,never smoked,drink a lot
2,female,right handed,city,tried smoking,drink a lot
4,female,right handed,village,tried smoking,social drinker
5,male,right handed,city,never smoked,never


Вроде все категориальные фичи действительно категориальные

### Автоматизируем процесс

In [92]:
numeric_selector = make_column_selector(dtype_include=np.number)    #type: ignore
categorical_selector = make_column_selector(dtype_include=object)   #type: ignore

## Предобработаем фичи

#### Числовые

Заполняем пропуски и стандартизируем

In [93]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#### Категориальные

Заполняем пропуски и кодируем

In [94]:
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

#### Собираем всю предобработку вместе

In [95]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numeric_transformer, make_column_selector(dtype_include=np.number)),  #type: ignore
        ("category", categorical_transformer, make_column_selector(dtype_include=["object", "category"]))   #type: ignore
    ]
)

### Фичеинжениринг

К текущим признакам:
- добавить PCA (на первые 5 компонент),
- SVD (на первые 5 компонент)


In [96]:
feature_union = FeatureUnion(
    transformer_list=[
        ("identity", "passthrough"),    #type: ignore        # исходные признаки как есть
        ("pca", PCA(n_components=5)),
        ("svd", TruncatedSVD(n_components=5)),
    ]
)

#### Итоговое преобразование признаков

In [97]:
features = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("features", feature_union),
])

features

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x16886a000>),
                                                 ('category',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x168c0ec30>)])),
                ('features',
                 FeatureUnion(transformer_list=[('identity', 'passthrough'),
                                                ('pca', PCA(n_components=5)),
                                                ('svd',
                                                 TruncatedSVD(n_components=5))]))])

In [98]:
features.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x16886a000>),
                                                 ('category',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x168c0ec30>)])),
                ('features',
                 FeatureUnion(transformer_list=[('identity',
                                                 FunctionTransformer(feature_names_out='one-to-one')),
                                                ('pca', PCA(n_components=5)),
                                                ('svd',
                                                 TruncatedSVD(n_components=5))]))])

## Модель

In [99]:
classifier_pipeline = Pipeline(
    steps=[('features', features),
           ('classifier', LogisticRegression(max_iter=1000, n_jobs=-1, class_weight="balanced"))]
)

classifier_pipeline

Pipeline(steps=[('features',
                 Pipeline(steps=[('preprocessor',
                                  ColumnTransformer(transformers=[('numerical',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(strategy='median')),
                                                                                   ('scaler',
                                                                                    StandardScaler())]),
                                                                   <sklearn.compose._column_transformer.make_column_selector object at 0x16886a000>),
                                                                  ('category',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(strategy='most_frequen...
                                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                                                   <sklearn.compose._column_transformer.make_column_selector object at 0x168c0ec30>)])),
                                 ('features',
                                  FeatureUnion(transformer_list=[('identity',
                                                                  FunctionTransformer(feature_names_out='one-to-one')),
                                                                 ('pca',
                                                                  PCA(n_components=5)),
                                                                 ('svd',
                                                                  TruncatedSVD(n_components=5))]))])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    n_jobs=-1))])

In [100]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=2)

In [101]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((539, 20), (135, 20), (539,), (135,))

In [102]:
classifier_pipeline.fit(X_train, y_train)

Pipeline(steps=[('features',
                 Pipeline(steps=[('preprocessor',
                                  ColumnTransformer(transformers=[('numerical',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(strategy='median')),
                                                                                   ('scaler',
                                                                                    StandardScaler())]),
                                                                   <sklearn.compose._column_transformer.make_column_selector object at 0x16886a000>),
                                                                  ('category',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(strategy='most_frequen...
                                                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                                                   <sklearn.compose._column_transformer.make_column_selector object at 0x168c0ec30>)])),
                                 ('features',
                                  FeatureUnion(transformer_list=[('identity',
                                                                  FunctionTransformer(feature_names_out='one-to-one')),
                                                                 ('pca',
                                                                  PCA(n_components=5)),
                                                                 ('svd',
                                                                  TruncatedSVD(n_components=5))]))])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    n_jobs=-1))])

In [103]:
preds = classifier_pipeline.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

          -1       0.53      0.65      0.58        48
           0       0.56      0.32      0.41        69
           1       0.24      0.50      0.33        18

    accuracy                           0.46       135
   macro avg       0.44      0.49      0.44       135
weighted avg       0.51      0.46      0.46       135



## Подбор параметров

In [104]:
f1 = make_scorer(f1_score, average="macro")

In [121]:
param_grid = {
    # PCA внутри FeatureUnion
    "features__features__pca__n_components": [5, 10, 15],
    # SVD внутри FeatureUnion
    "features__features__svd__n_components": [5, 10, 15],
    # Логистическая регрессия
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    'features__preprocessor__numerical__imputer__strategy':['median', 'mean']
}

In [122]:
grid = GridSearchCV(
    estimator=classifier_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring=f1,   # или "accuracy", "roc_auc_ovr" и т.п.
    n_jobs=-1,
    verbose=0
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('features',
                                        Pipeline(steps=[('preprocessor',
                                                         ColumnTransformer(transformers=[('numerical',
                                                                                          Pipeline(steps=[('imputer',
                                                                                                           SimpleImputer(strategy='median')),
                                                                                                          ('scaler',
                                                                                                           StandardScaler())]),
                                                                                          <sklearn.compose._column_transformer.make_column_selector object at 0x16886a000>),
                                                                                         ('category',
                                                                                          Pipeline(steps=[('imputer',
                                                                                                           SimpleIm...
                                        LogisticRegression(class_weight='balanced',
                                                           max_iter=1000,
                                                           n_jobs=-1))]),
             n_jobs=-1,
             param_grid={'classifier__C': [0.01, 0.1, 1.0, 10.0],
                         'features__features__pca__n_components': [5, 10, 15],
                         'features__features__svd__n_components': [5, 10, 15],
                         'features__preprocessor__numerical__imputer__strategy': ['median',
                                                                                  'mean']},
             scoring=make_scorer(f1_score, response_method='predict', average=macro))

In [123]:
grid.best_score_

0.42496725770073